In [1]:
import os
import ee
import geemap
from datetime import datetime, timedelta
import pandas as pd
import time
import shutil
from ipywidgets import widgets
from tkinter import Tk, filedialog


In [2]:
class SelectFilesButton(widgets.Button):
    """A file widget that leverages tkinter.filedialog."""
    def __init__(self):
        super(SelectFilesButton, self).__init__()
        self.add_traits(files=widgets.traitlets.List())
        self.description = "Select Files"
        self.icon = "square-o"
        self.style.button_color = "orange"
        self.on_click(self.select_files)

    @staticmethod
    def select_files(b):
        root = Tk()
        root.withdraw()
        root.call('wm', 'attributes', '.', '-topmost', True)
        b.files = filedialog.askopenfilename(multiple=True, filetypes=[('shp', '.shp')])
        b.description = "Files Selected"
        b.icon = "check-square-o"
        b.style.button_color = "lightgreen"

def _last_day_of_month(any_day):
    next_month = any_day.replace(day=28) + timedelta(days=4)  
    return next_month - timedelta(days=next_month.day)

def monthlist(begin, end):
    result = []
    while True:
        if begin.month == 12:
            next_month = begin.replace(year=begin.year + 1, month=1, day=1)
        else:
            next_month = begin.replace(month=begin.month + 1, day=1)
        if next_month > end:
            break
        result.append([begin.strftime("%Y-%m-%d"), _last_day_of_month(begin).strftime("%Y-%m-%d")])
        begin = next_month
    result.append([begin.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d")])
    return result

def date_format_conversion(date, output_format='%Y/%m/%d'):
    if date is None:
        return None
    try:
        parsed_date = datetime.strptime(date, output_format)
    except ValueError:
        try:
            parsed_date = datetime.strptime(date, '%Y%m%d')
        except ValueError:
            try:
                parsed_date = datetime.strptime(date, '%Y_%m_%d')
            except ValueError:
                try:
                    parsed_date = datetime.strptime(date, '%Y-%m-%d')
                except ValueError as e:
                    return logging.error(f'Unparsable date format {e}')
    return parsed_date.strftime(output_format)

# Function to get land cover from Dynamic World V1
def get_dynamic_world_landcover(image):
    land_cover = image.select([
        'water', 'trees', 'grass', 'flooded_vegetation',
        'crops', 'shrub_and_scrub', 'built', 'bare', 'snow_and_ice'
    ])
    return image.addBands(land_cover)


In [3]:
def date_format_concersion(date, output_format='%Y/%m/%d'):

    # Fool-proof: check if the input date is None
    if date is None:
        return None

    try: 
        parsed_date = datetime.strptime(date, output_format)
        
    except ValueError as e:

        try:
            # Try to parse the input date
            parsed_date = datetime.strptime(date, '%Y%m%d')
        except ValueError as e:
        
            try:
                parsed_date = datetime.strptime(date, '%Y_%m_%d')
            except ValueError as e:

                try:
                    parsed_date = datetime.strptime(date, '%Y-%m-%d')
                except ValueError as e:

                    return logging.error(f'Unparsable date format {e}')

    output_date = parsed_date.strftime(output_format)

    return output_date

In [4]:
#J
ee.Authenticate()

Enter verification code:  4/1AeanS0YWAto0q_cKcD3qQUe-HtQoftRqHSUXOCUJvcq0IdQxO62ylj2-Thk



Successfully saved authorization token.


In [5]:
# After executing this line of code for the first use, you can get the authentication number linked to Google.
Map = geemap.Map()
# Authenticate the Google earth engine with google account
ee.Initialize() 

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_0JLhFqfSY1uiEaW?source=Init


In [6]:

# Create date pickers for the user
star = widgets.DatePicker(description='Pick a Start Date', disabled=False)
end = widgets.DatePicker(description='Pick an End Date', disabled=False)
widgets.HBox([star, end])


In [7]:
#J try to use era5 method 20241030
# success

##list file
import os
import glob

# Replace 'your_directory_path' with the actual path
#directory_J = '/media/dyclab/新增磁碟區/Jeffery/data/shp/EU_100km_fishnet_simple_by_distance/EU_100km_fishnet_simple_by_distance.shp'
directory_J = "D:/EU_100km_fishnet_simple_by_distance/EU_100km_fishnet_simple_by_distance/EU_100km_fishnet_simple_by_distance.shp"

## 環境變數的 list
# 原始的清單
original_list = [
    "water",
    "trees",
    "grass",
    "flooded_vegetation",
    "crops",
    "shrub_and_scrub",
    "built",
    "bare",
    "snow_and_ice"
]



# 將每個元素轉換為單一元素的元組，並存入新的清單
env_var_list = [(item,) for item in original_list]
#band_name = ('temperature_2m',)

#--------------------------------------------------------------
# statics list
statics_OwO =['MEAN','MAXIMUM', 'MINIMUM', 'MEDIAN', 'STD', 'VARIANCE', 'SUM']
#statics = 'MEAN'


#-----------------------------------------------
file_name = 'Id'

#---------------------------------------------------------

# create folder
folder_name = 'dynamic_world_data'

# create folder name : data_all
if os.path.isdir(folder_name) == True:
    shutil.rmtree(folder_name, ignore_errors=True)
    os.makedirs(folder_name)
else:
    os.makedirs(folder_name)

#-------------------------------------------------------------

import time

#shpfile_OwO = '/media/dyclab/新增磁碟區/Jeffery/data/shp/EU_100km_fishnet_simple_by_distance/EU_100km_fishnet_simple_by_distance.shp'
shpfile_OwO = "D:/EU_100km_fishnet_simple_by_distance/EU_100km_fishnet_simple_by_distance/EU_100km_fishnet_simple_by_distance.shp"

for band_name in env_var_list:
    for statics in statics_OwO[0:1]:
        start_time = time.time()
        time_list = monthlist(star.value, end.value)
        states = geemap.shp_to_ee(shpfile_OwO)
        for i in range(0, len(time_list)):
                dynamic_world = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1') \
                .filter(ee.Filter.date(datetime.strptime(time_list[i][0], "%Y-%m-%d"), datetime.strptime(time_list[i][1], "%Y-%m-%d")+timedelta(days=1))) \
                .map(lambda image: image.select(list(band_name))) \
                .map(lambda image: image.clip(states)) \
                .map(lambda image: image.reproject(crs='EPSG:4326'))

                #改動
                if dynamic_world.size().getInfo() > 0:
                    dynamic_world_monthly_avg = dynamic_world.mean()
                else:
                    print("No images found for the given date range.")
                    continue
                #---------------------------
                out_dir = os.path.expanduser(folder_name)
                out_dem_stats = os.path.join(
                    out_dir, 'dynamic_world_{}_{}.csv'.format(statics, time_list[i]))

                if not os.path.exists(out_dir):
                    os.makedirs(out_dir)

                geemap.zonal_statistics(
                dynamic_world_monthly_avg, states, out_dem_stats, statistics_type=statics, scale=1000)
                
                
                data_temp = pd.read_csv(out_dem_stats)

                data = []
                
                # 初始化 df 為 data_temp 的副本
                df = data_temp.copy()


                mean_OwO = 'mean'
                
                if mean_OwO in data_temp.columns:
                    # Extract data
                    df[file_name] = data_temp.loc[:, [file_name]]
                    df[mean_OwO] = data_temp.loc[:, [mean_OwO]]

                    df.insert(0, 'Date', '')
                    df['Date'] = date_format_concersion(
                        time_list[i][0], output_format='%Y/%m/%d')

                    df.insert(1, 'Doy', '')
                    df['Doy'] = datetime.strptime(
                        time_list[i][0], '%Y-%m-%d').strftime('%j')

                    # Rename columns

                    band_name_OwO = ''.join(band_name)
                    df = df.rename(columns={'mean': band_name_OwO})

                    data.append(df)
                
                # 如果都是空直, 填入na
                else:
                    import numpy as np
                    df[file_name] = data_temp.loc[:, [file_name]]
                    
                    df.insert(0, mean_OwO, np.nan)

                    df.insert(0, 'Date', '')
                    df['Date'] = date_format_concersion(
                        time_list[i][0], output_format='%Y/%m/%d')

                    df.insert(1, 'Doy', '')
                    df['Doy'] = datetime.strptime(
                        time_list[i][0], '%Y-%m-%d').strftime('%j')

                    # Rename columns

                    band_name_OwO = ''.join(band_name)
                    df = df.rename(columns={'mean': band_name_OwO})

                    data.append(df)
                

                # 檢查 data 是否為空，再執行合併
                if data:
                    appended_data = pd.concat(data, axis=0, ignore_index=True)
                    # Output the file with date and doy back
                    appended_data.to_csv(out_dem_stats, index=False)
                else:
                    print("No data to concatenate")
                

                
        end_time = time.time()
        # 計算時間差
        elapsed_time = end_time - start_time
        # 列印這次迴圈的執行時間
        print(f"迴圈執行時間: {elapsed_time:.4f} 秒")



        def _rename_columns(df, column_name, suffix):
            if column_name in df.columns:
                df.rename(columns={column_name: f"{column_name}_{suffix}"}, inplace=True)

        def cbind_dynamic_world(statics):

            all_files = glob.glob(os.path.join(folder_name , "dynamic_world_{}*.csv".format(statics)))

            df_from_each_file = (pd.read_csv(f, sep = ",") for f in all_files)
            df_merged = pd.concat(df_from_each_file, ignore_index = True)

            for column_name in df_merged.columns:
                if column_name in band_name:
                    _rename_columns(df_merged, column_name, str(statics))

            band_name_str = ', '.join(band_name)

            df_merged.to_csv('D:/EU_result/dynamic_world_result/' + 
                             band_name_str +
                             '_' +
                             statics +
                             '_' +
                             str(star.value) +
                             '_' +
                             'to' +
                             '_' +
                             str(end.value) +
                             '.csv', index=False)


        cbind_dynamic_world(statics)

                

Computing statistics ...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\吳宏達\dynamic_world_data\dynamic_world_MEAN_['2023-01-01', '2023-01-31'].csv
Computing statistics ...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\吳宏達\dynamic_world_data\dynamic_world_MEAN_['2023-02-01', '2023-02-28'].csv
Computing statistics ...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\吳宏達\dynamic_world_data\dynamic_world_MEAN_['2023-03-01', '2023-03-31'].csv
Computing statistics ...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\吳宏達\dynamic_world_data\dynamic_world_MEAN_['2023-04-01', '2023-04-30'].csv
Computing statistics ...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\吳宏達\dynamic_world_data\dynamic_world_MEAN_['2023-05-01', '2023-05-31'].csv
Computing statistics ...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\吳宏達\dynamic_world_data\dynamic_world_MEAN_['2023-06-01', '2023-06-30'].csv
Computing statistics .